# Geometra: extracción de reseñas de Google Maps

**Autor: Aarón Eduardo Arceo Arjona**
___

Notebook complementario del Trabajo Final. Documenta cómo se obtuvieron las reseñas: la API de Places entrega como
máximo 5 reseñas por lugar, por lo que el texto se extrajo de Google Maps con Selenium y Chrome, con fines
exclusivamente académicos.

**Antes de ejecutar**

- **Requiere ejecución local** (Jupyter o VS Code) con Google Chrome instalado. Colab no tiene un navegador con
  ventana para iniciar sesión: ahí el notebook no extrae y muestra el resultado original de la muestra.
- Por defecto solo extrae una muestra de 3 cafeterías (`EXTRAER_TODO = False`), en alrededor de un minuto. Las
  1,230 cafeterías con reseñas tomaron varias horas.
- La entrada es `places_merida_clean.csv`, la base limpia que produce la sección 2 del notebook central.
- La salida completa, `reviews_scraped.csv`, es la que usa el notebook central (sección 4). Por practicidad ya está
  en la carpeta de datos de Google Drive, que este notebook descarga igual que el central. No se guarda el nombre
  de las personas autoras.
- Iniciar sesión en Google es opcional; los pasos están en la siguiente celda.

### Cómo iniciar sesión en Google para la extracción

Durante la extracción original, sin sesión iniciada Google Maps mostraba solo unas 10 reseñas por lugar, por lo que
se hizo con sesión para llegar a 25. En una prueba posterior la muestra obtuvo 25 sin sesión: el comportamiento de
Google Maps varía, y si se obtienen menos reseñas de las esperadas conviene iniciar sesión. Para hacerlo:

1. Cambiar `INICIAR_SESION = True` en la sección 0 y ejecutar el notebook.
2. En la celda *Inicio de sesión* se abre una ventana de Chrome en la página de acceso de Google. Iniciar sesión
   con una cuenta, de preferencia secundaria.
3. Volver al notebook y presionar Enter en el cuadro de texto que aparece debajo de la celda.
4. La sesión queda guardada en la carpeta `~/.geometra_perfil_chrome`, fuera del proyecto, y las siguientes
   ejecuciones entran con sesión aunque Chrome no se muestre. Basta hacerlo una vez; después se puede regresar a
   `INICIAR_SESION = False`.

Esa carpeta es de uso personal.

## 0. Configuración y datos

Selenium descarga por sí solo el controlador que corresponde a la versión de Chrome instalada.

In [1]:
import importlib.util
import subprocess
import sys

PAQUETES = {'selenium': 'selenium', 'gdown': 'gdown'}
faltantes = [pip for modulo, pip in PAQUETES.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltantes], check=True, capture_output=True)
print('Bibliotecas instaladas en esta sesión:', ', '.join(faltantes) or 'ninguna')

Bibliotecas instaladas en esta sesión: ninguna


In [2]:
# Parámetros -----------------------------------------------------------------
ID_DRIVE = '1WM18VwFvPEIWJGDBrt8Jndqw3WjfoGgk'   # ID de Datos_Geometra.zip en Google Drive
EXTRAER_TODO = False     # True: las 1,230 cafeterías con reseñas (varias horas)
N_MUESTRA = 3            # cafeterías de la muestra
INICIAR_SESION = False   # True: abre Chrome para iniciar sesión en Google (ver pasos arriba)
HEADLESS = True          # False: muestra la ventana de Chrome durante la extracción
MAX_RESENAS = 25         # tope de reseñas por lugar (las primeras que carga Google Maps)
SEMILLA = 0

import csv
import json
import random
import re
import time
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import (ElementClickInterceptedException, NoSuchElementException,
                                        TimeoutException, WebDriverException)
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

pd.set_option('display.max_columns', None, 'display.width', None, 'display.max_colwidth', 100)

EN_COLAB = 'google.colab' in sys.modules
RUTA_PERFIL_CHROME = Path.home() / '.geometra_perfil_chrome'   # perfil dedicado, guarda la sesión de Google

# Control de carga: pausas aleatorias para no saturar el servicio
PAUSA_MIN, PAUSA_MAX = 3.0, 7.0
PAUSA_LARGA_CADA, PAUSA_LARGA_MIN, PAUSA_LARGA_MAX = 50, 30.0, 60.0
USER_AGENT = ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
              '(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36')

# Selectores de Google Maps: pueden cambiar si Google actualiza su página
SELECTORS = {
    'review_block': 'div.jftiEf',
    'rating': '.kvMYJc',
    'date': '.rsqaWe',
    'text': '.wiI7pd',
    'more_button': 'button.w8nwRe',
}
print('Entorno:', 'Colab (no se extrae)' if EN_COLAB else 'local')

Entorno: local


In [3]:
def preparar_datos(carpeta='Datos_Geometra', id_drive=ID_DRIVE):
    """Devuelve la carpeta de datos del proyecto.

    Si la carpeta ya existe junto al notebook se usa tal cual; si no, se descarga el
    archivo comprimido desde Google Drive y se descomprime en el directorio actual.
    """
    ruta = Path(carpeta)
    if not ruta.is_dir():
        import gdown
        archivo = gdown.download(id=id_drive, output=f'{carpeta}.zip', quiet=True)
        with zipfile.ZipFile(archivo) as z:
            z.extractall('.')
    return ruta


DATOS = preparar_datos()
SALIDAS = Path('salidas')
SALIDAS.mkdir(exist_ok=True)
print('Datos en', DATOS.as_posix())

Datos en Datos_Geometra


## 1. Cafeterías a extraer

Se excluyen las que no tienen reseñas en Google. La muestra incluye una cafetería con miles de reseñas (para ver
que el desplazamiento llegue a 25), una con pocas (para ver que se detenga cuando ya no hay más) y el resto al azar.

In [4]:
lugares = pd.read_csv(DATOS / 'places_merida_clean.csv', dtype={'place_id': str, 'celda_id': str})
lugares['reviews_uri'] = lugares['google_maps_links'].apply(lambda s: json.loads(s)['reviewsUri'])   # abre el panel de reseñas
total_lugares = len(lugares)
lugares = lugares[lugares['n_resenas'] > 0].reset_index(drop=True)
print(f'Cafeterías en la base limpia: {total_lugares} | con reseñas en Google: {len(lugares)}')

extremos = pd.concat([lugares.nlargest(1, 'n_resenas'), lugares[lugares['n_resenas'].between(1, 3)].head(1)])
muestra = pd.concat([extremos, lugares.drop(extremos.index).sample(max(N_MUESTRA - len(extremos), 0),
                                                                   random_state=SEMILLA)]).head(N_MUESTRA)
a_extraer = lugares if EXTRAER_TODO else muestra
print(f'Cafeterías a extraer: {len(a_extraer)} ({"todas" if EXTRAER_TODO else "muestra"})')
a_extraer[['nombre_google', 'rating', 'n_resenas']].head(10)

Cafeterías en la base limpia: 1619 | con reseñas en Google: 1230
Cafeterías a extraer: 3 (muestra)


,nombre_google,rating,n_resenas
0,Café La Habana,4.0,7006.0
913,Malteadas y frappes la 42 x 21,3.7,3.0
19,La Libertad García Lavín | Desayunos & Brunch Mérida Yucatán,4.7,1939.0


## 2. Navegador e inicio de sesión

In [5]:
def crear_driver(headless=HEADLESS):
    """Abre Chrome con el perfil dedicado del proyecto, que conserva la sesión de Google entre ejecuciones."""
    opciones = Options()
    if headless:
        opciones.add_argument('--headless=new')
    opciones.add_argument(f'--user-data-dir={RUTA_PERFIL_CHROME}')
    opciones.add_argument('--window-size=1366,900')
    opciones.add_argument('--lang=es-MX')
    opciones.add_argument(f'user-agent={USER_AGENT}')
    opciones.add_argument('--disable-notifications')
    opciones.add_argument('--disable-blink-features=AutomationControlled')
    opciones.add_experimental_option('excludeSwitches', ['enable-automation'])
    opciones.add_experimental_option('prefs', {'profile.managed_default_content_settings.images': 2})   # sin imágenes
    driver = webdriver.Chrome(options=opciones)
    driver.set_page_load_timeout(30)
    return driver

In [6]:
# Inicio de sesión (solo con INICIAR_SESION = True)
if INICIAR_SESION and not EN_COLAB:
    driver_login = crear_driver(headless=False)
    driver_login.get('https://accounts.google.com/')
    input('Inicie sesión en la ventana de Chrome y presione Enter aquí al terminar...')
    driver_login.quit()
    print('Sesión guardada en ~/.geometra_perfil_chrome')
else:
    print('Se usa el perfil tal como está: con sesión si ya se inició antes; si no, Google Maps puede '
          'mostrar menos reseñas por lugar.')

Se usa el perfil tal como está: con sesión si ya se inició antes; si no, Google Maps puede mostrar menos reseñas por lugar.


## 3. Extracción

Para cada cafetería se abre el enlace directo a su panel de reseñas, se desplaza el panel hasta reunir
`MAX_RESENAS` o hasta que ya no cargan más, se expanden los textos truncados y se leen la calificación, la fecha
relativa y el texto. Cada lugar se guarda al terminar y un archivo de avance permite reanudar la corrida completa;
los fallos se registran sin detener el proceso.

In [7]:
# Sube desde una reseña ya cargada hasta el primer contenedor con desplazamiento vertical real
_JS_BUSCAR_SCROLLABLE = """
let nodo = arguments[0].parentElement;
while (nodo && nodo !== document.body) {
    const estilo = window.getComputedStyle(nodo);
    if ((estilo.overflowY === 'auto' || estilo.overflowY === 'scroll') &&
        nodo.scrollHeight > nodo.clientHeight + 10) {
        return nodo;
    }
    nodo = nodo.parentElement;
}
return null;
"""
_JS_SCROLL_Y_MEDIR = """
arguments[0].scrollTop = arguments[0].scrollHeight;
return [arguments[0].scrollHeight, document.querySelectorAll(arguments[1]).length];
"""


def _texto_seguro(elemento, selector_css):
    """Texto de un subelemento, o None si no existe."""
    try:
        return elemento.find_element(By.CSS_SELECTOR, selector_css).text.strip() or None
    except NoSuchElementException:
        return None


def _rating_seguro(elemento, selector_css):
    """Número de estrellas leído de la etiqueta accesible del subelemento, o None."""
    try:
        nodo = elemento.find_element(By.CSS_SELECTOR, selector_css)
        match = re.search(r'(\d+([.,]\d+)?)', nodo.get_attribute('aria-label') or nodo.text or '')
        return float(match.group(1).replace(',', '.')) if match else None
    except NoSuchElementException:
        return None


def extraer_resenas_de_lugar(driver, url, max_resenas=MAX_RESENAS, tiempo_espera=15, intentos_max=5):
    """Abre el panel de reseñas de un lugar y devuelve hasta `max_resenas` reseñas.

    Google Maps carga las reseñas en tandas al detectar desplazamiento, así que se desplaza
    el panel hasta reunir `max_resenas` o hasta que, en `intentos_max` intentos seguidos,
    no crezcan ni el número de reseñas ni la altura del panel.
    """
    driver.get(url)
    primer_bloque = WebDriverWait(driver, tiempo_espera).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, SELECTORS['review_block'])))
    panel = (driver.execute_script(_JS_BUSCAR_SCROLLABLE, primer_bloque)
             or driver.execute_script('return document.scrollingElement || document.body;'))

    vistos, alto_visto, sin_avance = 0, 0, 0
    while vistos < max_resenas and sin_avance < intentos_max:
        time.sleep(1.0)
        alto, bloques = driver.execute_script(_JS_SCROLL_Y_MEDIR, panel, SELECTORS['review_block'])
        sin_avance = 0 if (bloques > vistos or alto > alto_visto) else sin_avance + 1
        vistos, alto_visto = bloques, max(alto_visto, alto)

    bloques = driver.find_elements(By.CSS_SELECTOR, SELECTORS['review_block'])[:max_resenas]
    for bloque in bloques:   # expandir textos truncados ("Más") antes de leerlos
        try:
            driver.execute_script('arguments[0].click();', bloque.find_element(By.CSS_SELECTOR, SELECTORS['more_button']))
        except (NoSuchElementException, ElementClickInterceptedException):
            pass
    return [{'rating': _rating_seguro(b, SELECTORS['rating']),
             'relative_date': _texto_seguro(b, SELECTORS['date']),
             'review_text': _texto_seguro(b, SELECTORS['text'])} for b in bloques]

In [8]:
COLUMNAS_RESENAS = ['place_id', 'place_name', 'rating', 'relative_date', 'review_text', 'scraped_at']
COLUMNAS_ERRORES = ['place_id', 'place_name', 'url', 'error', 'scraped_at']


def _append_csv(ruta, columnas, filas):
    """Agrega filas a un CSV, escribiendo el encabezado si el archivo es nuevo."""
    existe = ruta.exists()
    with open(ruta, 'a', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=columnas)
        if not existe:
            writer.writeheader()
        writer.writerows(filas)


def _resumen_error(e):
    """Tipo de excepción y primera línea del mensaje (sin el detalle técnico del controlador)."""
    mensaje = str(e).strip().splitlines()[0].strip() if str(e).strip() else ''
    return f'{type(e).__name__}: {mensaje}' if mensaje else type(e).__name__


def procesar_lugares(df, driver, ruta_resenas, ruta_avance, ruta_errores):
    """Extrae las reseñas de cada lugar pendiente, guardando cada uno al terminar.

    `ruta_avance` lista los place_id ya resueltos (con o sin error) para poder reanudar.
    """
    hechos = set(ruta_avance.read_text(encoding='utf-8').splitlines()) if ruta_avance.exists() else set()
    pendientes = df[~df['place_id'].isin(hechos)].reset_index(drop=True)
    print(f'Pendientes: {len(pendientes)} de {len(df)}')

    for i, fila in pendientes.iterrows():
        ahora = datetime.now(timezone.utc).isoformat()
        try:
            resenas = extraer_resenas_de_lugar(driver, fila['reviews_uri'])
            _append_csv(ruta_resenas, COLUMNAS_RESENAS, [
                {'place_id': fila['place_id'], 'place_name': fila['nombre_google'], **r, 'scraped_at': ahora}
                for r in resenas])
            print(f'[{i + 1}/{len(pendientes)}] {fila["nombre_google"]}: {len(resenas)} reseñas')
        except Exception as e:
            _append_csv(ruta_errores, COLUMNAS_ERRORES, [{'place_id': fila['place_id'], 'place_name': fila['nombre_google'],
                                                         'url': fila['reviews_uri'], 'error': _resumen_error(e),
                                                         'scraped_at': ahora}])
            print(f'[{i + 1}/{len(pendientes)}] {fila["nombre_google"]}: error ({_resumen_error(e)})')
        finally:
            with open(ruta_avance, 'a', encoding='utf-8') as f:
                f.write(f'{fila["place_id"]}\n')

        if i + 1 < len(pendientes):
            larga = (i + 1) % PAUSA_LARGA_CADA == 0
            time.sleep(random.uniform(PAUSA_LARGA_MIN, PAUSA_LARGA_MAX) if larga else random.uniform(PAUSA_MIN, PAUSA_MAX))

In [9]:
sufijo = '' if EXTRAER_TODO else '_muestra'
RUTA_RESENAS = SALIDAS / ('reviews_scraped.csv' if EXTRAER_TODO else 'reviews_muestra.csv')
RUTA_AVANCE = SALIDAS / f'ids_procesados{sufijo}.txt'
RUTA_ERRORES = SALIDAS / f'errores{sufijo}.csv'
if not EXTRAER_TODO:
    for ruta in (RUTA_RESENAS, RUTA_AVANCE, RUTA_ERRORES):
        ruta.unlink(missing_ok=True)   # la muestra se extrae de nuevo en cada ejecución

driver = None
if EN_COLAB:
    print('En Colab no se extrae: se muestran las reseñas originales de la muestra.')
else:
    try:
        driver = crear_driver()
    except WebDriverException as e:
        print(f'No se pudo abrir Chrome; revise que esté instalado. {_resumen_error(e)}')
    if driver is not None:
        try:
            procesar_lugares(a_extraer, driver, RUTA_RESENAS, RUTA_AVANCE, RUTA_ERRORES)
        finally:
            driver.quit()
SE_EXTRAJO = driver is not None

Pendientes: 3 de 3
[1/3] Café La Habana: 25 reseñas
[2/3] Malteadas y frappes la 42 x 21: 3 reseñas
[3/3] La Libertad García Lavín | Desayunos & Brunch Mérida Yucatán: 25 reseñas


## 4. Resultado

Se compara con la extracción original (septiembre de 2026, con sesión iniciada). Los números pueden diferir, ya
que Google Maps cambia con el tiempo las reseñas que muestra y su orden.

In [10]:
original = pd.read_csv(DATOS / 'reviews_scraped.csv', dtype={'place_id': str})
original_muestra = original[original['place_id'].isin(a_extraer['place_id'])]
if not SE_EXTRAJO:
    extraidas = original_muestra
    print('Fuente: extracción original (reviews_scraped.csv en Drive).')
elif RUTA_RESENAS.exists():
    extraidas = pd.read_csv(RUTA_RESENAS, dtype={'place_id': str})
    print(f'Fuente: extracción de esta ejecución ({RUTA_RESENAS.as_posix()}).')
else:
    extraidas = pd.DataFrame(columns=COLUMNAS_RESENAS)
    print('Esta ejecución no obtuvo reseñas; revise los errores registrados.')

comparacion = (a_extraer[['place_id', 'nombre_google', 'n_resenas']]
               .assign(extraidas_ahora=lambda d: d['place_id'].map(extraidas.groupby('place_id').size()).fillna(0).astype(int),
                       en_el_original=lambda d: d['place_id'].map(original_muestra.groupby('place_id').size()).fillna(0).astype(int))
               .drop(columns='place_id'))
print(f'Reseñas: {len(extraidas)} | con texto: {extraidas["review_text"].notna().sum()}')
if RUTA_ERRORES.exists():
    print(f'Lugares con error: {len(pd.read_csv(RUTA_ERRORES))}')
display(comparacion.head(10))

extraidas.assign(review_text=extraidas['review_text'].str.slice(0, 120))[
    ['place_name', 'rating', 'relative_date', 'review_text']].head(10)

Fuente: extracción de esta ejecución (salidas/reviews_muestra.csv).
Reseñas: 53 | con texto: 53


,nombre_google,n_resenas,extraidas_ahora,en_el_original
0,Café La Habana,7006.0,25,25
913,Malteadas y frappes la 42 x 21,3.0,3,3
19,La Libertad García Lavín | Desayunos & Brunch Mérida Yucatán,1939.0,25,25


,place_name,rating,relative_date,review_text
0,Café La Habana,1.0,Editado Hace 2 meses,El servicio amable pero nos comentaron que tuvieron problemas con algo en la cocina y nuestro de...
1,Café La Habana,5.0,Hace 6 meses,No puedo contar la cantidad de veces que he ido a comer ahí con mi familia. Es un favorito recur...
2,Café La Habana,5.0,Hace 3 meses,"Excelente experiencia. Alberto nos brindó una atención excepcional, siempre amable, atento y pen..."
3,Café La Habana,5.0,Hace 5 meses,"Es un restaurante cafetería que evoca la vieja Habana, por lo que parece detenida en el tiempo. ..."
4,Café La Habana,5.0,Hace 4 meses,Excelente atención de todos en especial Samantha buenas recomendaciones y todo el tiempo al pend...
5,Café La Habana,5.0,Hace 2 meses,"De los mejores lugares en el centro: sin falso hype, con gran sabor y buenas porciones. Aquí no ..."
6,Café La Habana,5.0,Hace 3 meses,"Excelente y amable servicio del mesero Manuel Ojeda, la comida deliciosa y el tiempo de espera b..."
7,Café La Habana,4.0,Hace 5 meses,Muy buen lugar para ir a desayunar la verdad muy agradable el ambiente es un buen lugar para rel...
8,Café La Habana,2.0,Hace 9 meses,"Parece un restaurante de los años 80, la arquitectura es muy buena, buen ambiente, no así su com..."
9,Café La Habana,5.0,Hace 3 meses,"El lugar es bastante ameno y tiene un sazón muy rico , ya lo e recomendado varias veces y con ju..."


El notebook central usa el archivo completo `reviews_scraped.csv`: 17,422 reseñas de 1,223 cafeterías (7 lugares
fallaron), de las cuales 15,685 tienen texto.